[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/31.%20The%20Rail-Terminal%20Scheduling%20Problem/P31-Tier-4_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 31. The Rail-Terminal Scheduling Problem

## Tier 4: Ensemble Learning for Makespan Prediction

### Goal

Build an ensemble-learning pipeline that extracts features from a rail-terminal instance, trains several regression models to predict a makespan target, and combines their predictions with a meta-learner. The pipeline also reports a simple crane-to-task allocation and diagnostic information about model disagreement.

This is a predictive-learning demonstration, not an optimization algorithm. Its output is not guaranteed to be an optimal makespan or a feasible timed schedule.

### Inputs and outputs

The instance input consists of tasks, cranes, and task attributes such as processing time, position, earliest start, deadline, and predecessors. The training pipeline uses feature vectors and target makespans. In this notebook, those targets are generated synthetically by a formula; they are not obtained by solving the scheduling problem.

For an input instance, the trained ensemble returns a predicted makespan, predictions from its base regressors, a model-disagreement band, Random Forest feature importances, and a task allocation across cranes. The allocation lists tasks by crane; it does not specify start times, travel times, or a conflict-free operational schedule.

### Scope and modeling assumptions

- Synthetic training targets are constructed from a workload lower-bound term, a complexity-based overhead, and random noise. They are illustrative regression targets, not verified optimal solutions.
- The synthetic task generator creates precedence links within each train, but the feature vector summarizes precedence rather than encoding a full scheduling model.
- Crane movement, interference between cranes, train-track geometry, setup times, and explicit deadline feasibility are not modeled in the prediction target or the reported task allocation.
- The four base learners all predict the same scalar target. Their names identify the model families used in the ensemble; they do not indicate that the models independently assign cranes, enforce constraints, or tune a schedule.
- The reported disagreement band is based on the spread of the base predictions. It is a diagnostic, not a statistically calibrated confidence interval or a guarantee of prediction coverage.

### Ensemble method

Let an instance be represented by a feature vector \(x\), with synthetic target \(y\). Four base regressors produce predictions \(f_1(x),\ldots,f_4(x)\). A meta-learner \(g\) is trained on predictions for a held-out portion of the training data and estimates the target as

$$
\hat{y}(x) = g\bigl(f_1(x), f_2(x), f_3(x), f_4(x)\bigr).
$$

The training instances are first divided into a base-model subset and a meta-model subset. The base regressors are fitted on the first subset; their predictions on the second subset become the meta-learner's input features. An outer validation set is kept separate for evaluation. This separation gives the meta-learner predictions from instances that were not used to fit the base regressors.

The feature scaler is fitted only on the base-model subset. This keeps information from the meta-model subset and the outer validation set out of the scaler's fitted statistics.

### What to examine

- Compare the validation errors of the individual regressors and the meta-learner. An ensemble is not guaranteed to outperform every base model.
- Inspect feature importances as Random Forest split-based importance scores, not as causal effects.
- Treat disagreement among base predictions as a warning signal rather than as a calibrated probability statement.
- Distinguish makespan prediction from schedule construction: the predicted scalar does not certify feasibility or optimality.

### Concrete example context

The demonstration instance contains 18 tasks, 5 trains, and 4 cranes. Its task and crane attributes are explicitly defined in the data cell. The notebook does not assume a separate historical dataset of 500 solved instances or report predetermined accuracy figures; it generates synthetic training examples and computes evaluation results when the cells are run.

In [ ]:
from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, List, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class Task:
    """Describe one container-handling task in the example instance."""

    id: int
    train_id: int
    processing_time: int
    position: int
    earliest_start: int
    deadline: int
    predecessors: List[int]


@dataclass
class Crane:
    """Describe a crane by its identifier and initial position."""

    id: int
    initial_position: int


@dataclass
class EnsemblePrediction:
    """Collect a prediction and its associated diagnostic information."""

    predicted_makespan: float
    confidence_interval: Tuple[float, float]
    base_predictions: Dict[str, float]
    feature_importance: Dict[str, float]
    assigned_schedule: Dict[int, List[int]]


def create_ml_example_instance() -> Tuple[Dict[int, Task], Dict[int, Crane]]:
    """Create the fixed example instance used for feature extraction and prediction."""
    tasks = {
        1: Task(1, 1, 12, 2, 0, 40, []),
        2: Task(2, 1, 8, 3, 0, 40, [1]),
        3: Task(3, 1, 10, 4, 0, 40, [1]),
        4: Task(4, 1, 7, 5, 0, 40, [2, 3]),
        5: Task(5, 2, 9, 6, 12, 52, []),
        6: Task(6, 2, 11, 7, 12, 52, [5]),
        7: Task(7, 2, 8, 8, 12, 52, [5]),
        8: Task(8, 3, 13, 9, 24, 64, []),
        9: Task(9, 3, 9, 10, 24, 64, [8]),
        10: Task(10, 3, 10, 11, 24, 64, [8]),
        11: Task(11, 4, 8, 12, 36, 76, []),
        12: Task(12, 4, 12, 13, 36, 76, [11]),
        13: Task(13, 4, 7, 14, 36, 76, [11]),
        14: Task(14, 5, 10, 15, 48, 88, []),
        15: Task(15, 5, 9, 16, 48, 88, [14]),
        16: Task(16, 5, 11, 17, 48, 88, [14]),
        17: Task(17, 5, 6, 18, 48, 88, [15, 16]),
        18: Task(18, 5, 8, 19, 48, 88, [15, 16]),
    }

    cranes = {
        1: Crane(1, 1),
        2: Crane(2, 6),
        3: Crane(3, 11),
        4: Crane(4, 16),
    }
    return tasks, cranes


tasks, cranes = create_ml_example_instance()

print("Example instance:")
print(f"  Total tasks: {len(tasks)}")
print(f"  Total trains: {len({task.train_id for task in tasks.values()})}")
print(f"  Total cranes: {len(cranes)}")
print("  Precedence links and parallel branches are represented in the task data.")

train_task_counts = defaultdict(int)
for task in tasks.values():
    train_task_counts[task.train_id] += 1

print("\nTask distribution by train:")
for train_id, count in sorted(train_task_counts.items()):
    print(f"  Train {train_id}: {count} tasks")

## 1. Feature engineering

A feature is a numerical summary that allows a regression model to use information about an instance. Here, the features describe instance size, temporal windows, processing times, positions, precedence structure, and workload. They do not encode every detail needed to verify a schedule.

For example, the feature vector includes the task count \(n\), crane count \(m\), total workload \(W = \sum_i p_i\), and precedence density. The code also computes means, standard deviations, ranges, and ratios. These summaries are useful to the learning pipeline because the regressors require fixed-length numerical inputs, but summarization can discard important structural information.

The resulting vector has a fixed ordering given by `FEATURE_NAMES`. Feature extraction and prediction must use this same ordering so that each model input column retains its intended meaning.

In [ ]:
def extract_problem_features(
    task_dict: Dict[int, Task],
    crane_dict: Dict[int, Crane],
) -> np.ndarray:
    """Convert task and crane data into a fixed-length numerical feature vector."""
    features = []

    num_tasks = len(task_dict)
    num_cranes = len(crane_dict)
    num_trains = len({task.train_id for task in task_dict.values()})
    features.extend([num_tasks, num_cranes, num_trains])

    # Task-level summaries are weighted by the number of tasks on each train.
    train_arrivals = [task.earliest_start for task in task_dict.values()]
    train_deadlines = [task.deadline for task in task_dict.values()]
    if train_arrivals:
        features.extend([
            np.mean(train_arrivals),
            np.std(train_arrivals),
            np.min(train_arrivals),
            np.max(train_arrivals),
            np.max(train_arrivals) - np.min(train_arrivals),
        ])
        features.extend([
            np.mean(train_deadlines),
            np.std(train_deadlines),
            np.min(train_deadlines),
            np.max(train_deadlines),
            np.mean([
                deadline - arrival
                for deadline, arrival in zip(train_deadlines, train_arrivals)
            ]),
        ])
    else:
        features.extend([0] * 10)

    processing_times = [task.processing_time for task in task_dict.values()]
    if processing_times:
        minimum_processing = np.min(processing_times)
        features.extend([
            np.mean(processing_times),
            np.std(processing_times),
            minimum_processing,
            np.max(processing_times),
            np.max(processing_times) / minimum_processing
            if minimum_processing > 0
            else 0,
        ])
    else:
        features.extend([0] * 5)

    positions = [task.position for task in task_dict.values()]
    crane_positions = [crane.initial_position for crane in crane_dict.values()]
    if positions and crane_positions:
        features.extend([
            np.mean(positions),
            np.std(positions),
            np.max(positions) - np.min(positions),
            np.mean(crane_positions),
            np.std(crane_positions),
            np.max(crane_positions) - np.min(crane_positions),
        ])
    else:
        features.extend([0] * 6)

    total_predecessors = sum(len(task.predecessors) for task in task_dict.values())
    tasks_with_predecessors = sum(
        bool(task.predecessors) for task in task_dict.values()
    )
    precedence_density = tasks_with_predecessors / num_tasks if num_tasks else 0
    features.extend([
        total_predecessors,
        tasks_with_predecessors,
        precedence_density,
    ])

    total_work = sum(task.processing_time for task in task_dict.values())
    work_per_crane = total_work / num_cranes if num_cranes else 0
    work_per_train = total_work / num_trains if num_trains else 0
    features.extend([total_work, work_per_crane, work_per_train])

    tasks_per_crane = num_tasks / num_cranes if num_cranes else 0
    tasks_per_train = num_tasks / num_trains if num_trains else 0
    features.extend([
        tasks_per_crane,
        tasks_per_train,
        num_tasks * num_cranes * num_trains,
    ])

    return np.asarray(features, dtype=float)


FEATURE_NAMES = [
    "num_tasks", "num_cranes", "num_trains",
    "avg_arrival", "std_arrival", "min_arrival", "max_arrival", "arrival_spread",
    "avg_deadline", "std_deadline", "min_deadline", "max_deadline", "avg_window",
    "avg_processing", "std_processing", "min_processing", "max_processing", "processing_ratio",
    "avg_position", "std_position", "position_spread",
    "avg_crane_pos", "std_crane_pos", "crane_pos_spread",
    "total_predecessors", "tasks_with_predecessors", "precedence_density",
    "total_work", "work_per_crane", "work_per_train",
    "tasks_per_crane", "tasks_per_train", "complexity_metric",
]

features = extract_problem_features(tasks, cranes)
assert len(features) == len(FEATURE_NAMES)

print(f"Extracted {len(features)} features from the example instance.")
print(f"Feature vector shape: {features.shape}")
print("First ten features:")
for name, value in zip(FEATURE_NAMES[:10], features[:10]):
    print(f"  {name}: {value:.2f}")

## 2. Synthetic training data

A supervised regression model learns from pairs \((x_j, y_j)\): a feature vector \(x_j\) and a numerical target \(y_j\). Since this demonstration does not load solved historical instances, it constructs synthetic targets. The target formula combines a workload term, a complexity-based overhead, and random noise:

$$
y_j = \max\left(\frac{W_j}{m_j} + H_j + \varepsilon_j,\; 1.1\frac{W_j}{m_j}\right),
$$

where \(W_j\) is the total processing time, \(m_j\) is the number of cranes, \(H_j\) is the constructed overhead, and \(\varepsilon_j\) is random noise. This formula defines a teaching target; it does not solve the scheduling problem and does not establish an optimal makespan.

The generated instances are split into a training set and an outer validation set. The validation set is reserved for evaluating predictions after fitting. Synthetic targets and random instance generation are useful for demonstrating the mechanics of the pipeline, but performance on them is not evidence of real-world accuracy.

In [ ]:
def generate_synthetic_training_data(
    num_instances: int = 500,
) -> Tuple[List[np.ndarray], List[float]]:
    """Generate feature vectors and illustrative synthetic makespan targets."""
    print(f"Generating {num_instances} synthetic training instances...")

    X_features = []
    y_makespans = []

    for instance_id in range(num_instances):
        num_trains = random.randint(2, 6)
        num_cranes = random.randint(2, 4)
        tasks_per_train = random.randint(2, 4)

        instance_tasks = {}
        task_id = 1
        for train_id in range(1, num_trains + 1):
            arrival_time = (train_id - 1) * random.randint(8, 16)
            deadline = arrival_time + random.randint(30, 50)

            for task_num in range(tasks_per_train):
                processing_time = random.randint(5, 15)
                position = random.randint(1, 20)
                predecessors = []
                if task_num > 0 and random.random() < 0.7:
                    predecessors = [task_id - 1]

                instance_tasks[task_id] = Task(
                    id=task_id,
                    train_id=train_id,
                    processing_time=processing_time,
                    position=position,
                    earliest_start=arrival_time,
                    deadline=deadline,
                    predecessors=predecessors,
                )
                task_id += 1

        instance_cranes = {
            crane_id: Crane(crane_id, random.randint(1, 20))
            for crane_id in range(1, num_cranes + 1)
        }

        X_features.append(
            extract_problem_features(instance_tasks, instance_cranes)
        )

        total_processing = sum(
            task.processing_time for task in instance_tasks.values()
        )
        theoretical_workload_bound = total_processing / num_cranes
        complexity_overhead = (
            0.2 * len(instance_tasks)
            + 0.1 * np.std([
                task.position for task in instance_tasks.values()
            ])
            + 0.15 * sum(
                len(task.predecessors) for task in instance_tasks.values()
            )
        )
        noise = random.gauss(0, theoretical_workload_bound * 0.05)
        synthetic_target = max(
            theoretical_workload_bound + complexity_overhead + noise,
            theoretical_workload_bound * 1.1,
        )
        y_makespans.append(synthetic_target)

        if (instance_id + 1) % 100 == 0:
            print(f"  Generated {instance_id + 1}/{num_instances} instances")

    print(f"Generated {len(X_features)} synthetic instances.")
    return X_features, y_makespans


X_all, y_all = generate_synthetic_training_data(500)
X_train, X_val, y_train, y_val = train_test_split(
    X_all,
    y_all,
    test_size=0.2,
    random_state=42,
)

print("\nData split:")
print(f"  Training set: {len(X_train)} instances")
print(f"  Validation set: {len(X_val)} instances")
print(f"  Feature dimension: {len(X_train[0])}")
print(f"  Synthetic target range in training set: {min(y_train):.1f} to {max(y_train):.1f}")

## 3. Train the base learners and meta-learner

The base learners are four regressors with different model structures: Random Forest, Gradient Boosting, support vector regression (SVR), and a multilayer perceptron (MLP). Each predicts the same synthetic makespan target. The meta-learner is a second Random Forest regressor that learns how to combine the four base predictions.

The training function divides its input training data into two parts. Base learners are fitted on the first part. Their predictions on the second part train the meta-learner. The separate outer validation data is used only for evaluation. This training order is the defining stacking step in this ensemble: the meta-learner receives base-model predictions for instances not used to fit those base models.

The models named `crane_assigner`, `constraint_handler`, and `parameter_tuner` retain the illustrative roles used in the framework's organization, but all three are scalar regressors of the same target. They do not produce crane assignments, enforce constraints, or tune scheduling parameters.

In [ ]:
class RailTerminalEnsemble:
    """Stack four regressors to predict the synthetic makespan target."""

    def __init__(self):
        self.makespan_predictor = RandomForestRegressor(
            n_estimators=100,
            random_state=42,
            max_depth=10,
        )
        self.crane_assigner = GradientBoostingRegressor(
            n_estimators=100,
            random_state=42,
            max_depth=6,
        )
        self.constraint_handler = SVR(
            kernel="rbf",
            C=1.0,
            gamma="scale",
        )
        self.parameter_tuner = MLPRegressor(
            hidden_layer_sizes=(64, 32),
            random_state=42,
            max_iter=500,
        )
        self.meta_learner = RandomForestRegressor(
            n_estimators=50,
            random_state=42,
            max_depth=6,
        )
        self.feature_scaler = StandardScaler()
        self.is_trained = False
        self.model_performance = {}

    def train_ensemble(
        self,
        X_features: List[np.ndarray],
        y_makespans: List[float],
        X_validation: List[np.ndarray],
        y_validation: List[float],
    ) -> Dict[str, float]:
        """Fit the stacked regressors and evaluate them on held-out data."""
        print("Training the ensemble learning framework...")

        X = np.asarray(X_features, dtype=float)
        y = np.asarray(y_makespans, dtype=float)
        X_val_array = np.asarray(X_validation, dtype=float)
        y_val_array = np.asarray(y_validation, dtype=float)

        if X.ndim != 2 or X.shape[1] != len(FEATURE_NAMES):
            raise ValueError("Training features do not match the feature definition.")
        if X_val_array.ndim != 2 or X_val_array.shape[1] != X.shape[1]:
            raise ValueError("Validation features must have the same columns as training features.")
        if len(X) != len(y) or len(X_val_array) != len(y_val_array):
            raise ValueError("Each feature row must have exactly one target value.")

        X_base, X_meta, y_base, y_meta = train_test_split(
            X,
            y,
            test_size=0.3,
            random_state=42,
        )

        # Fit preprocessing only on the data used to fit the base regressors.
        X_base_scaled = self.feature_scaler.fit_transform(X_base)
        X_meta_scaled = self.feature_scaler.transform(X_meta)
        X_val_scaled = self.feature_scaler.transform(X_val_array)

        print(f"  Base learners: {len(X_base)} instances")
        print(f"  Meta-learner: {len(X_meta)} instances")
        print(f"  Outer validation: {len(X_val_array)} instances")

        base_models = {
            "makespan": self.makespan_predictor,
            "gradient_boosting": self.crane_assigner,
            "svr": self.constraint_handler,
            "neural_network": self.parameter_tuner,
        }

        validation_base_predictions = {}
        meta_training_predictions = []
        for model_name, model in base_models.items():
            model.fit(X_base_scaled, y_base)
            meta_training_predictions.append(model.predict(X_meta_scaled))
            validation_base_predictions[model_name] = model.predict(X_val_scaled)

        meta_training_matrix = np.column_stack(meta_training_predictions)
        self.meta_learner.fit(meta_training_matrix, y_meta)

        validation_meta_matrix = np.column_stack(
            list(validation_base_predictions.values())
        )
        validation_ensemble_predictions = self.meta_learner.predict(
            validation_meta_matrix
        )

        self.model_performance = {}
        for model_name, predictions in validation_base_predictions.items():
            self.model_performance[f"{model_name}_mae"] = mean_absolute_error(
                y_val_array, predictions
            )
            self.model_performance[f"{model_name}_r2"] = r2_score(
                y_val_array, predictions
            )

        self.model_performance["meta_mae"] = mean_absolute_error(
            y_val_array, validation_ensemble_predictions
        )
        self.model_performance["meta_r2"] = r2_score(
            y_val_array, validation_ensemble_predictions
        )
        self.is_trained = True

        print("\nValidation metrics:")
        for model_name in base_models:
            print(
                f"  {model_name}: "
                f"MAE={self.model_performance[f'{model_name}_mae']:.3f}, "
                f"R²={self.model_performance[f'{model_name}_r2']:.3f}"
            )
        print(
            f"  Meta-learner: MAE={self.model_performance['meta_mae']:.3f}, "
            f"R²={self.model_performance['meta_r2']:.3f}"
        )
        print("These metrics evaluate predictions of the synthetic target.")
        return self.model_performance

    def predict_schedule(
        self,
        task_dict: Dict[int, Task],
        crane_dict: Dict[int, Crane],
    ) -> EnsemblePrediction:
        """Predict a makespan and return a simple crane-to-task allocation."""
        if not self.is_trained:
            raise ValueError("Train the ensemble before requesting predictions.")

        features = extract_problem_features(task_dict, crane_dict)
        features_scaled = self.feature_scaler.transform(features.reshape(1, -1))

        base_predictions = {
            "RandomForest": float(
                self.makespan_predictor.predict(features_scaled)[0]
            ),
            "GradientBoosting": float(
                self.crane_assigner.predict(features_scaled)[0]
            ),
            "SVR": float(self.constraint_handler.predict(features_scaled)[0]),
            "NeuralNetwork": float(
                self.parameter_tuner.predict(features_scaled)[0]
            ),
        }
        base_prediction_values = np.asarray(list(base_predictions.values()))
        predicted_makespan = float(
            self.meta_learner.predict(base_prediction_values.reshape(1, -1))[0]
        )

        # This band summarizes model disagreement; it is not a calibrated interval.
        prediction_spread = float(np.std(base_prediction_values))
        disagreement_band = (
            predicted_makespan - 1.96 * prediction_spread,
            predicted_makespan + 1.96 * prediction_spread,
        )

        feature_importance = dict(
            zip(FEATURE_NAMES, self.makespan_predictor.feature_importances_)
        )
        assigned_schedule = self._generate_schedule_from_prediction(
            task_dict,
            crane_dict,
            predicted_makespan,
        )

        return EnsemblePrediction(
            predicted_makespan=predicted_makespan,
            confidence_interval=disagreement_band,
            base_predictions=base_predictions,
            feature_importance=feature_importance,
            assigned_schedule=assigned_schedule,
        )

    def _generate_schedule_from_prediction(
        self,
        task_dict: Dict[int, Task],
        crane_dict: Dict[int, Crane],
        target_makespan: float,
    ) -> Dict[int, List[int]]:
        """Allocate tasks to cranes by processing load; this is not a timed schedule."""
        del target_makespan  # A scalar makespan estimate does not determine assignments.
        allocation = {crane_id: [] for crane_id in crane_dict}
        crane_loads = {crane_id: 0 for crane_id in crane_dict}

        # A deterministic priority order makes the illustrative allocation repeatable.
        task_ids = sorted(
            task_dict,
            key=lambda task_id: (
                -task_dict[task_id].processing_time,
                task_id,
            ),
        )
        for task_id in task_ids:
            best_crane = min(
                crane_loads,
                key=lambda crane_id: (crane_loads[crane_id], crane_id),
            )
            allocation[best_crane].append(task_id)
            crane_loads[best_crane] += task_dict[task_id].processing_time

        return allocation


ensemble = RailTerminalEnsemble()
performance_metrics = ensemble.train_ensemble(
    X_train,
    y_train,
    X_val,
    y_val,
)

print("\nValidation MAE summary:")
for model_name, metric in performance_metrics.items():
    if model_name.endswith("_mae"):
        print(f"  {model_name}: {metric:.3f} synthetic-target units")

## 4. Predict on the example instance

After training, feature extraction applies the same feature definitions and ordering used during training. The scaler transforms the new feature vector, each base regressor predicts the target, and the meta-learner combines those predictions.

The returned crane allocation is a load-balancing illustration based on processing times. It assigns each task once, but it does not schedule start times or enforce precedence, travel, interference, arrival, or deadline constraints. Therefore, it must not be interpreted as a feasible rail-terminal schedule.

In [ ]:
print("Testing the ensemble on the example instance:")
print(f"  Problem: {len(tasks)} tasks, {len(cranes)} cranes")

prediction = ensemble.predict_schedule(tasks, cranes)

print("\nEnsemble prediction:")
print(f"  Predicted synthetic makespan target: {prediction.predicted_makespan:.2f}")
print(
    "  Base-model disagreement band: "
    f"[{prediction.confidence_interval[0]:.2f}, "
    f"{prediction.confidence_interval[1]:.2f}]"
)
print("  The band is diagnostic and is not a calibrated confidence interval.")

print("\nBase-model predictions:")
for model_name, value in prediction.base_predictions.items():
    print(f"  {model_name}: {value:.2f}")

print("\nIllustrative crane-to-task allocation (not a timed schedule):")
for crane_id, task_list in prediction.assigned_schedule.items():
    total_processing = sum(tasks[task_id].processing_time for task_id in task_list)
    print(
        f"  Crane {crane_id}: tasks {task_list}; "
        f"assigned processing-time total = {total_processing}"
    )

allocated_task_ids = [
    task_id
    for task_list in prediction.assigned_schedule.values()
    for task_id in task_list
]
assert sorted(allocated_task_ids) == sorted(tasks)
assert len(allocated_task_ids) == len(set(allocated_task_ids))

## 5. Feature importance

The fitted Random Forest exposes impurity-based feature importances. These scores describe how much the fitted trees used each feature to reduce their split criteria. They are model-dependent associations, not causal explanations, and correlated features can share or distort importance.

The plots below show the largest individual scores and aggregate scores into broad feature categories. The category totals are descriptive summaries of the Random Forest importances.

In [ ]:
def analyze_feature_importance(
    ensemble_prediction: EnsemblePrediction,
) -> List[Tuple[str, float]]:
    """Print and plot the fitted Random Forest feature-importance scores."""
    sorted_features = sorted(
        ensemble_prediction.feature_importance.items(),
        key=lambda item: item[1],
        reverse=True,
    )

    print("Random Forest feature importance:")
    for rank, (feature_name, importance) in enumerate(sorted_features[:10], start=1):
        print(f"  {rank:2d}. {feature_name:25s}: {importance:.4f}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    top_features = sorted_features[:15]
    feature_names = [item[0] for item in top_features]
    importances = [item[1] for item in top_features]

    bars = ax1.barh(feature_names[::-1], importances[::-1], color="skyblue", alpha=0.8)
    ax1.set_xlabel("Random Forest impurity-based importance")
    ax1.set_title("Top Feature Importance Scores")
    ax1.grid(True, axis="x", alpha=0.3)
    for bar, importance in zip(bars, importances[::-1]):
        ax1.text(
            bar.get_width() + 0.001,
            bar.get_y() + bar.get_height() / 2,
            f"{importance:.3f}",
            ha="left",
            va="center",
        )

    category_importance = defaultdict(float)
    for feature_name, importance in ensemble_prediction.feature_importance.items():
        if "arrival" in feature_name or "deadline" in feature_name or "window" in feature_name:
            category = "Temporal features"
        elif "crane" in feature_name:
            category = "Crane features"
        elif "position" in feature_name:
            category = "Position features"
        elif "precedence" in feature_name:
            category = "Precedence features"
        elif "work" in feature_name or "complexity" in feature_name:
            category = "Workload and size features"
        elif "processing" in feature_name or "task" in feature_name:
            category = "Task features"
        else:
            category = "Other size features"
        category_importance[category] += importance

    categories = list(category_importance)
    values = [category_importance[category] for category in categories]
    ax2.pie(
        values,
        labels=categories,
        autopct="%1.1f%%",
        startangle=90,
    )
    ax2.set_title("Importance Totals by Feature Category")

    plt.tight_layout()
    plt.show()
    return sorted_features


sorted_features = analyze_feature_importance(prediction)

## 6. Compare validation performance

Mean absolute error (MAE) summarizes the average absolute difference between a prediction and its target. Root mean squared error (RMSE) gives greater weight to large errors. The coefficient of determination \(R^2\) compares the model with a constant-mean baseline; it can be negative when predictions are worse than that baseline.

All reported metrics below are computed on the outer validation set and refer to the synthetic target. The comparison is descriptive: it does not establish performance on operational data or prove that stacking is superior.

In [ ]:
def compare_model_performance(
    fitted_ensemble: RailTerminalEnsemble,
    X_validation: List[np.ndarray],
    y_validation: List[float],
) -> Dict[str, Dict[str, float]]:
    """Compare base regressors and the meta-learner on held-out instances."""
    X_val_array = np.asarray(X_validation, dtype=float)
    y_val_array = np.asarray(y_validation, dtype=float)
    X_val_scaled = fitted_ensemble.feature_scaler.transform(X_val_array)

    base_predictions = {
        "Random Forest": fitted_ensemble.makespan_predictor.predict(X_val_scaled),
        "Gradient Boosting": fitted_ensemble.crane_assigner.predict(X_val_scaled),
        "SVR": fitted_ensemble.constraint_handler.predict(X_val_scaled),
        "Neural Network": fitted_ensemble.parameter_tuner.predict(X_val_scaled),
    }
    meta_inputs = np.column_stack(list(base_predictions.values()))
    base_predictions["Ensemble"] = fitted_ensemble.meta_learner.predict(meta_inputs)

    results = {}
    for model_name, predictions in base_predictions.items():
        results[model_name] = {
            "MAE": mean_absolute_error(y_val_array, predictions),
            "RMSE": np.sqrt(mean_squared_error(y_val_array, predictions)),
            "R2": r2_score(y_val_array, predictions),
        }

    comparison_table = pd.DataFrame(results).T
    print("Validation performance on the synthetic target:")
    display(comparison_table.round(3))

    model_names = list(results)
    colors = ["red", "orange", "gold", "green", "blue"]
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

    mae_values = [results[name]["MAE"] for name in model_names]
    bars = ax1.bar(model_names, mae_values, color=colors, alpha=0.8)
    ax1.set_ylabel("Mean absolute error (synthetic-target units)")
    ax1.set_title("Validation MAE")
    ax1.tick_params(axis="x", rotation=20)
    ax1.grid(True, axis="y", alpha=0.3)
    for bar, value in zip(bars, mae_values):
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.3f}",
            ha="center",
            va="bottom",
        )

    r2_values = [results[name]["R2"] for name in model_names]
    bars = ax2.bar(model_names, r2_values, color=colors, alpha=0.8)
    ax2.set_ylabel("R² score")
    ax2.set_title("Validation R²")
    ax2.tick_params(axis="x", rotation=20)
    ax2.grid(True, axis="y", alpha=0.3)
    for bar, value in zip(bars, r2_values):
        vertical_alignment = "bottom" if value >= 0 else "top"
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            value,
            f"{value:.3f}",
            ha="center",
            va=vertical_alignment,
        )

    y_min = float(np.min(y_val_array))
    y_max = float(np.max(y_val_array))
    ax3.scatter(
        y_val_array,
        base_predictions["Ensemble"],
        alpha=0.65,
        label="Ensemble",
    )
    ax3.scatter(
        y_val_array,
        base_predictions["Random Forest"],
        alpha=0.45,
        label="Random Forest",
    )
    ax3.plot([y_min, y_max], [y_min, y_max], "k--", label="Perfect prediction")
    ax3.set_xlabel("Synthetic target")
    ax3.set_ylabel("Predicted target")
    ax3.set_title("Predicted versus synthetic target")
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    ensemble_errors = np.abs(base_predictions["Ensemble"] - y_val_array)
    forest_errors = np.abs(base_predictions["Random Forest"] - y_val_array)
    ax4.hist(ensemble_errors, bins=20, alpha=0.65, label="Ensemble", density=True)
    ax4.hist(forest_errors, bins=20, alpha=0.5, label="Random Forest", density=True)
    ax4.set_xlabel("Absolute error (synthetic-target units)")
    ax4.set_ylabel("Density")
    ax4.set_title("Validation absolute-error distributions")
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return results


performance_results = compare_model_performance(ensemble, X_val, y_val)

## 7. Prediction and workload diagnostics

The total processing workload divided by the number of cranes is a simple lower bound on elapsed processing time under the idealized assumption that cranes can work in parallel without interference. It is not, by itself, a feasible makespan or a complete lower bound for every operational model. Comparing the ensemble prediction with this quantity is therefore a diagnostic, not a schedule-efficiency calculation.

The prediction time measured below is specific to the current machine, software environment, and instance. It should not be presented as a general deployment guarantee. The base-model spread describes disagreement among four regressors trained on the same synthetic target; it does not quantify all sources of uncertainty.

In [ ]:
def comprehensive_performance_analysis(
    ensemble_prediction: EnsemblePrediction,
    task_dict: Dict[int, Task],
    crane_dict: Dict[int, Crane],
    training_features: List[np.ndarray],
    sorted_feature_importance: List[Tuple[str, float]],
    fitted_ensemble: RailTerminalEnsemble,
) -> None:
    """Summarize prediction diagnostics and the scope of the example."""
    predicted_target = ensemble_prediction.predicted_makespan
    interval_width = (
        ensemble_prediction.confidence_interval[1]
        - ensemble_prediction.confidence_interval[0]
    )
    base_values = np.asarray(
        list(ensemble_prediction.base_predictions.values()),
        dtype=float,
    )

    print("Prediction and workload diagnostics:")
    print(f"  Predicted synthetic target: {predicted_target:.2f}")
    print(f"  Base-model standard deviation: {np.std(base_values):.2f}")
    print(f"  Disagreement-band width: {interval_width:.2f}")
    print("  The disagreement band is not a calibrated confidence interval.")

    total_processing = sum(
        task.processing_time for task in task_dict.values()
    )
    workload_reference = total_processing / len(crane_dict) if crane_dict else 0
    print(f"\n  Total task processing time: {total_processing}")
    print(f"  Workload-per-crane reference value: {workload_reference:.2f}")
    print("  This reference does not certify schedule feasibility or optimality.")

    print("\nMost important Random Forest features:")
    for rank, (feature_name, importance) in enumerate(
        sorted_feature_importance[:5],
        start=1,
    ):
        print(f"  {rank}. {feature_name}: {importance:.4f}")

    training_complexity = len(training_features) * len(FEATURE_NAMES)
    print("\nTraining-data dimensions:")
    print(f"  Training instances: {len(training_features)}")
    print(f"  Features per instance: {len(FEATURE_NAMES)}")
    print(f"  Instance-feature entries: {training_complexity:,}")

    start_time = time.perf_counter()
    for _ in range(100):
        fitted_ensemble.predict_schedule(task_dict, crane_dict)
    average_prediction_time_ms = (
        (time.perf_counter() - start_time) / 100
    ) * 1000
    print(f"\n  Average prediction-call time in this run: {average_prediction_time_ms:.3f} ms")
    print("  This measurement is environment-dependent and is not a deployment guarantee.")


comprehensive_performance_analysis(
    prediction,
    tasks,
    cranes,
    X_train,
    sorted_features,
    ensemble,
)

## 8. Interpretation and limitations

### How this learning tier differs from an optimization method

An optimization method searches for a decision that satisfies a mathematical model and minimizes an objective. This ensemble instead learns a mapping from instance summaries to a numerical target. It can provide a quick estimate after training, but a prediction is not a certificate that a corresponding schedule exists or is optimal.

### Strengths of the demonstrated ensemble

- **Multiple model structures:** the base regressors can represent different patterns in the feature-to-target relationship.
- **Stacked combination:** the meta-learner uses the base predictions as inputs and learns a second-stage regression.
- **Consistent feature interface:** a fixed feature order allows training and prediction to share the same representation.
- **Diagnostics:** validation metrics, feature-importance scores, and base-model disagreement offer several ways to inspect the fitted models.

These are properties of the pipeline, not evidence that the ensemble outperforms a particular optimization method or individual model. Such a conclusion requires suitable independently generated test instances and valid reference targets.

### Limitations

- The synthetic target is generated by a formula rather than by solving a scheduling model or observing verified historical outcomes.
- The feature summaries omit details that can determine feasibility, including complete precedence structure, crane travel, and crane interference.
- The displayed crane allocation is a simple load-balancing assignment, not a time-indexed schedule and not a constraint-aware optimization result.
- Feature importance is associational and model-specific.
- The disagreement band is not calibrated and should not be interpreted as a 95% confidence interval.
- Generalization to real operations requires representative data, carefully defined target values, and evaluation against relevant operational criteria.

### Natural extensions

A larger study could replace synthetic targets with verified outcomes from the scheduling method under study, evaluate on independent instances, and enrich the instance representation with operationally relevant data. Any schedule intended for deployment would also require explicit feasibility checks against the applicable constraints.